In [ ]:
"""This training workflow is intentionally separate from case inference. It can:
1. train and validate a signature-specific YOLO detector;
2. train a writer-disjoint Siamese ResNet-18 on BHSig260;
3. calibrate similarity boundaries on BHSig260 validation writers;
4. evaluate once on held-out BHSig260 test writers;
5. optionally evaluate untouched CEDAR generalization; and
6. promote reviewed artifacts to the canonical paths used by inference.
"""

In [ ]:
# # Model Training and Checkpoint Recreation
#
# This notebook is not needed for normal case analysis. Use it only when:
#
# - the YOLO detector must be retrained on improved complete-signature boxes;
# - the Siamese verifier must be rebuilt;
# - configuration or preprocessing changed incompatibly;
# - saved checkpoints were lost; or
# - a controlled model-development experiment is required.
#
# Training outputs first go into a versioned folder. Canonical production paths
# are replaced only in explicit promotion cells.

In [ ]:
# ## Training Cell 1 — Mount Drive, install dependencies, and verify GPU

# %%
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Signature-Forensic-Prototype"
)
CONFIG_PATH = PROJECT_DIRECTORY / "config.yaml"
REQUIREMENTS_PATH = PROJECT_DIRECTORY / "requirements.txt"

if not PROJECT_DIRECTORY.is_dir():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIRECTORY}"
    )
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"config.yaml not found: {CONFIG_PATH}"
    )

if REQUIREMENTS_PATH.is_file():
    package_arguments = ["-r", str(REQUIREMENTS_PATH)]
else:
    package_arguments = [
        "numpy>=1.26,<3",
        "PyYAML>=6.0,<7",
        "Pillow>=10.0,<12",
        "opencv-python-headless>=4.9,<5",
        "PyMuPDF>=1.24,<2",
        "scikit-image>=0.22,<1",
        "scipy>=1.11,<2",
        "ImageHash>=4.3,<5",
        "ultralytics>=8.3,<9",
        "torch>=2.2,<3",
        "torchvision>=0.17,<1",
        "transformers>=4.45,<6",
        "safetensors>=0.4,<1",
        "scikit-learn>=1.4,<2",
        "matplotlib>=3.8,<4",
    ]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *package_arguments,
    ],
    check=True,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. In Colab select Runtime → Change runtime "
        "type → GPU before training."
    )

DEVICE = "cuda"
YOLO_DEVICE = 0

print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA device: Tesla T4


In [ ]:
# ## Training Cell 2 — Create an isolated versioned training run
#
# Large extracted datasets and YOLO working files stay on the temporary Colab
# VM for speed. Checkpoints, histories, splits, metrics, and manifests are saved
# to Drive after each important stage.

# %%
from datetime import datetime, timezone
import json
import shutil

TRAINING_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

VM_TRAINING_ROOT = Path("/content/signature_model_training") / TRAINING_ID
VM_DATA_ROOT = VM_TRAINING_ROOT / "datasets"
VM_RUN_ROOT = VM_TRAINING_ROOT / "runs"

DRIVE_TRAINING_ROOT = (
    PROJECT_DIRECTORY
    / "models/training_runs"
    / TRAINING_ID
)
DRIVE_TRAINING_ROOT.mkdir(parents=True, exist_ok=False)
VM_DATA_ROOT.mkdir(parents=True, exist_ok=True)
VM_RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("Training ID:", TRAINING_ID)
print("Temporary VM workspace:", VM_TRAINING_ROOT)
print("Persistent Drive artifacts:", DRIVE_TRAINING_ROOT)

Training ID: 20260729T151920_289023Z
Temporary VM workspace: /content/signature_model_training/20260729T151920_289023Z
Persistent Drive artifacts: /content/drive/MyDrive/Signature-Forensic-Prototype/models/training_runs/20260729T151920_289023Z


In [ ]:
# ## Training Cell 3 — Import training APIs and inspect configuration
#
# The consolidated inference runner and this notebook share the same model
# architecture and preprocessing functions. This prevents a checkpoint from
# being trained with one transform and used with another.

# %%
import importlib
import sys
import yaml

if str(PROJECT_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIRECTORY))

for module_name in list(sys.modules):
    if (
        module_name == "src.detection"
        or module_name == "src.verification"
        or module_name == "src.external_validation"
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

from src.detection import (
    load_detection_config,
    load_yolo_model,
    train_detector,
    validate_detector,
)
from src.verification import (
    SiameseResNet18,
    assert_writer_disjoint,
    calibrate_similarity_thresholds,
    collect_pair_scores,
    create_writer_disjoint_split,
    discover_bhsig260,
    generate_balanced_pairs,
    load_siamese_checkpoint,
    load_verification_config,
    save_calibration,
    save_writer_split,
    summarize_writers,
    train_siamese_model,
)
from src.external_validation import (
    discover_cedar,
    evaluate_cedar_external,
)

detection_config = load_detection_config(CONFIG_PATH)
verification_config = load_verification_config(CONFIG_PATH)

print("YOLO configuration:")
print(yaml.safe_dump(detection_config, sort_keys=False))
print("Siamese configuration:")
print(yaml.safe_dump(verification_config, sort_keys=False))

# %% [markdown]
# # Part A — YOLO signature detector
#
# The YOLO dataset must contain complete document images and boxes covering the
# complete visible signature, including disconnected dots and flourishes.
#
# Expected archive content:
#
# ```text
# yolo_signature_dataset/
# ├── data.yaml
# ├── images/
# │   ├── train/
# │   └── val/
# └── labels/
#     ├── train/
#     └── val/
# ```
#
# Include negative pages with no signatures. A model trained only on cropped
# signatures cannot learn document-level localization.

YOLO configuration:
model_path: models/yolo_signature_detector.pt
base_model: yolo11n.pt
class_names:
  0: signature
confidence_threshold: 0.25
iou_threshold: 0.5
image_size: 1280
minimum_box_area_ratio: 0.0005
maximum_box_area_ratio: 0.35
boundary_margin_pixels: 3

Siamese configuration:
model_path: models/siamese_resnet18.pt
calibration_path: models/verification_calibration.json
writer_split_path: models/writer_split.json
training_history_path: models/siamese_training_history.json
embedding_dimension: 256
input_width: 224
input_height: 224
minimum_reference_count: 3
recommended_reference_count: 5
epochs: 10
batch_size: 32
learning_rate: 0.0001
weight_decay: 0.0001
contrastive_margin: 1.0
data_loader_workers: 2
positive_pairs_per_writer: 40
skilled_negative_pairs_per_writer: 20
random_negative_pairs_per_writer: 20
train_writer_fraction: 0.8
validation_writer_fraction: 0.1
uncertainty_margin: 0.05
provisional_consistent_similarity: 0.8
elevated_inconsistency_similarity: 0.55
imagenet_m

In [ ]:
# ## Training Cell 4 — Stage the YOLO dataset ZIP on the Colab VM
#
# Upload the ZIP to the configured Drive path before running this cell. Reading
# one archive and extracting locally is faster than training from thousands of
# individual Drive files.
# For YOLO training you need to download a raining dataset into training_inputs folder and rename it as yolo_signature_dataset.zip
# Various dataset you can use
# 1 - https://docs.ultralytics.com/datasets/detect/signature ( Used in current run)
# 2 - https://huggingface.co/datasets/tech4humans/signature-detection

In [ ]:
import zipfile

# EDIT THIS if your archive has a different name or location.
YOLO_DATASET_ZIP = (
    PROJECT_DIRECTORY
    / "training_inputs/yolo_signature_dataset.zip"
)

if not YOLO_DATASET_ZIP.is_file():
    raise FileNotFoundError(
        "YOLO dataset ZIP not found. Upload it to: "
        f"{YOLO_DATASET_ZIP}"
    )

YOLO_ZIP_LOCAL = VM_DATA_ROOT / YOLO_DATASET_ZIP.name
YOLO_EXTRACT_ROOT = VM_DATA_ROOT / "yolo"

shutil.copy2(YOLO_DATASET_ZIP, YOLO_ZIP_LOCAL)
with zipfile.ZipFile(YOLO_ZIP_LOCAL) as archive:
    archive.extractall(YOLO_EXTRACT_ROOT)

data_yaml_candidates = sorted(
    list(YOLO_EXTRACT_ROOT.rglob("data.yaml"))
    + list(YOLO_EXTRACT_ROOT.rglob("signature.yaml"))
)

if len(data_yaml_candidates) == 1:
    YOLO_DATA_YAML = data_yaml_candidates[0]
elif not data_yaml_candidates:
    # The official Ultralytics signature.zip may contain the images/labels
    # without embedding the separately maintained signature.yaml file. Find
    # the conventional dataset root and create an equivalent local YAML.
    train_image_directories = [
        path
        for path in YOLO_EXTRACT_ROOT.rglob("train")
        if path.is_dir()
        and path.parent.name == "images"
        and (path.parent.parent / "images/val").is_dir()
        and (path.parent.parent / "labels/train").is_dir()
        and (path.parent.parent / "labels/val").is_dir()
    ]
    if len(train_image_directories) != 1:
        raise ValueError(
            "No dataset YAML was found and the standard "
            "images/train, images/val, labels/train, labels/val layout "
            "could not be identified uniquely."
        )
    yolo_dataset_root = train_image_directories[0].parent.parent
    YOLO_DATA_YAML = yolo_dataset_root / "data.yaml"
    YOLO_DATA_YAML.write_text(
        yaml.safe_dump(
            {
                "path": str(yolo_dataset_root),
                "train": "images/train",
                "val": "images/val",
                "names": {0: "signature"},
            },
            sort_keys=False,
        ),
        encoding="utf-8",
    )
    print(
        "The ZIP had no YAML; created a local data.yaml for the "
        "standard Ultralytics directory layout."
    )
else:
    raise ValueError(
        "Expected at most one data.yaml/signature.yaml inside the YOLO "
        f"archive; found {len(data_yaml_candidates)}: {data_yaml_candidates}"
    )

print("YOLO dataset YAML:", YOLO_DATA_YAML)

YOLO dataset YAML: /content/signature_model_training/20260729T151920_289023Z/datasets/yolo/signature.yaml


In [ ]:
# ## Training Cell 5 — Validate YOLO dataset metadata and counts
#
# This is a structural check, not a complete annotation audit. Always visualize
# a sample of bounding boxes before expensive training.

YOLO_IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}

yolo_dataset_config = yaml.safe_load(
    YOLO_DATA_YAML.read_text(encoding="utf-8")
)
names_value = yolo_dataset_config.get("names")
names_text = str(names_value).lower()
if "signature" not in names_text:
    raise ValueError(
        "YOLO data.yaml must define a class named 'signature'."
    )
if "train" not in yolo_dataset_config or "val" not in yolo_dataset_config:
    raise ValueError(
        "YOLO data.yaml must define train and val image locations."
    )

yolo_images = [
    path
    for path in YOLO_EXTRACT_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in YOLO_IMAGE_EXTENSIONS
]
yolo_labels = list(YOLO_EXTRACT_ROOT.rglob("*.txt"))

if not yolo_images or not yolo_labels:
    raise ValueError(
        "The extracted YOLO dataset needs images and label text files."
    )

print("Images found:", len(yolo_images))
print("Label files found:", len(yolo_labels))
print("Classes:", names_value)
print(
    "Reminder: label count may be lower than image count when negative "
    "images intentionally have no signature boxes."
)

Images found: 178
Label files found: 179
Classes: {0: 'signature'}
Reminder: label count may be lower than image count when negative images intentionally have no signature boxes.


In [ ]:
# ## Training Cell 6 — Train and validate YOLO
#
# Training outputs remain on the VM. The best checkpoint and validation report
# are copied to the versioned Drive training directory.

# %%
YOLO_EPOCHS = 50
YOLO_BATCH_SIZE = 8

yolo_training = train_detector(
    data_yaml_path=YOLO_DATA_YAML,
    base_model=detection_config.get(
        "base_model",
        "yolo11n.pt",
    ),
    epochs=YOLO_EPOCHS,
    image_size=int(detection_config["image_size"]),
    batch_size=YOLO_BATCH_SIZE,
    project_directory=VM_RUN_ROOT / "yolo",
    run_name="signature_detector",
    device=YOLO_DEVICE,
    random_seed=42,
)

YOLO_VERSIONED_CHECKPOINT = (
    DRIVE_TRAINING_ROOT
    / "yolo_signature_detector.pt"
)
shutil.copy2(
    yolo_training["best_checkpoint"],
    YOLO_VERSIONED_CHECKPOINT,
)

versioned_yolo_model = load_yolo_model(
    YOLO_VERSIONED_CHECKPOINT
)
yolo_validation_report = validate_detector(
    model=versioned_yolo_model,
    data_yaml_path=YOLO_DATA_YAML,
    image_size=int(detection_config["image_size"]),
    confidence_threshold=float(
        detection_config["confidence_threshold"]
    ),
    device=YOLO_DEVICE,
)

YOLO_VALIDATION_REPORT_PATH = (
    DRIVE_TRAINING_ROOT
    / "yolo_validation_report.json"
)
YOLO_VALIDATION_REPORT_PATH.write_text(
    json.dumps(
        yolo_validation_report,
        indent=2,
    ),
    encoding="utf-8",
)

print("YOLO best checkpoint:", YOLO_VERSIONED_CHECKPOINT)
print("YOLO validation:")
print(json.dumps(yolo_validation_report, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/signature_model_training/20260729T151920_289023Z/datasets/yolo/signature.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

In [ ]:
# ## Training Cell 7 — Promote the reviewed YOLO checkpoint
#
# If you are fine with the YOLO training results  in Cell 6
# Mark ROMOTE_YOLO = True , below and run the code to save the model output in drive for future reference

# PROMOTE_YOLO = True
PROMOTE_YOLO = False

CANONICAL_YOLO_CHECKPOINT = (
    PROJECT_DIRECTORY
    / detection_config["model_path"]
)

if PROMOTE_YOLO:
    CANONICAL_YOLO_CHECKPOINT.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    if CANONICAL_YOLO_CHECKPOINT.is_file():
        backup_path = (
            DRIVE_TRAINING_ROOT
            / "previous_yolo_signature_detector.pt"
        )
        shutil.copy2(
            CANONICAL_YOLO_CHECKPOINT,
            backup_path,
        )
        print("Previous YOLO backed up:", backup_path)
    shutil.copy2(
        YOLO_VERSIONED_CHECKPOINT,
        CANONICAL_YOLO_CHECKPOINT,
    )
    shutil.copy2(
        YOLO_VALIDATION_REPORT_PATH,
        PROJECT_DIRECTORY
        / "models/yolo_validation_report.json",
    )
    print("YOLO promoted:", CANONICAL_YOLO_CHECKPOINT)
else:
    print(
        "YOLO was not promoted. Review the versioned checkpoint first:",
        YOLO_VERSIONED_CHECKPOINT,
    )

YOLO promoted: /content/drive/MyDrive/Signature-Forensic-Prototype/models/yolo_signature_detector.pt


In [ ]:
# # Part B — Siamese ResNet-18 on BHSig260
#
# BHSig260 writers are split before pairs are generated. This prevents the same
# writer from appearing in training, validation, and held-out testing.

# ## Training Cell 8 — Stage and discover BHSig260

# EDIT THIS if your archive has a different name or location.
BHSIG260_ZIP = (
    PROJECT_DIRECTORY
    / "training_inputs/BHSig260.zip"
)

if not BHSIG260_ZIP.is_file():
    raise FileNotFoundError(
        "BHSig260 ZIP not found. Upload it to: "
        f"{BHSIG260_ZIP}"
    )

BHSIG_ZIP_LOCAL = VM_DATA_ROOT / BHSIG260_ZIP.name
BHSIG_EXTRACT_ROOT = VM_DATA_ROOT / "bhsig260"

shutil.copy2(BHSIG260_ZIP, BHSIG_ZIP_LOCAL)
with zipfile.ZipFile(BHSIG_ZIP_LOCAL) as archive:
    archive.extractall(BHSIG_EXTRACT_ROOT)


def find_bhsig_script_directory(root, script_name):
    """Choose the deepest matching folder with the most signature images."""

    candidates = [
        path
        for path in root.rglob("*")
        if path.is_dir()
        and script_name.lower() in path.name.lower()
    ]
    ranked = []
    for path in candidates:
        image_count = sum(
            child.is_file()
            and child.suffix.lower() in YOLO_IMAGE_EXTENSIONS
            for child in path.rglob("*")
        )
        if image_count:
            ranked.append(
                (image_count, len(path.parts), path)
            )
    if not ranked:
        raise FileNotFoundError(
            f"Could not find the {script_name} BHSig260 folder."
        )
    return max(
        ranked,
        key=lambda item: (
            item[0],
            item[1],
        ),
    )[2]


BENGALI_DIRECTORY = find_bhsig_script_directory(
    BHSIG_EXTRACT_ROOT,
    "Bengali",
)
HINDI_DIRECTORY = find_bhsig_script_directory(
    BHSIG_EXTRACT_ROOT,
    "Hindi",
)

bhsig_writers = discover_bhsig260(
    BENGALI_DIRECTORY,
    HINDI_DIRECTORY,
)
bhsig_summary = summarize_writers(bhsig_writers)

print("Bengali directory:", BENGALI_DIRECTORY)
print("Hindi directory:", HINDI_DIRECTORY)
print("BHSig260 summary:")
print(json.dumps(bhsig_summary, indent=2))

Bengali directory: /content/signature_model_training/20260729T151920_289023Z/datasets/bhsig260/BHSig260-Bengali/BHSig260-Bengali
Hindi directory: /content/signature_model_training/20260729T151920_289023Z/datasets/bhsig260/BHSig260-Hindi/BHSig260-Hindi
BHSig260 summary:
{
  "writers": 260,
  "bengali_writers": 100,
  "hindi_writers": 160,
  "genuine_samples": 6240,
  "forged_samples": 7800
}


In [ ]:
# ## Training Cell 9 — Create writer-disjoint splits and balanced pairs
#
# Positive pairs are genuine/genuine from one writer. Negative pairs combine
# genuine/skilled-forgery and genuine/random-impostor examples.

writer_split = create_writer_disjoint_split(
    bhsig_writers,
    train_fraction=float(
        verification_config["train_writer_fraction"]
    ),
    validation_fraction=float(
        verification_config["validation_writer_fraction"]
    ),
    random_seed=42,
)
assert_writer_disjoint(writer_split)

WRITER_SPLIT_PATH = (
    DRIVE_TRAINING_ROOT / "writer_split.json"
)
save_writer_split(writer_split, WRITER_SPLIT_PATH)


def make_pairs(split_name, seed_offset):
    """Generate reproducible balanced pairs for one writer split."""

    return generate_balanced_pairs(
        writer_split[split_name],
        positive_pairs_per_writer=int(
            verification_config[
                "positive_pairs_per_writer"
            ]
        ),
        skilled_negative_pairs_per_writer=int(
            verification_config[
                "skilled_negative_pairs_per_writer"
            ]
        ),
        random_negative_pairs_per_writer=int(
            verification_config[
                "random_negative_pairs_per_writer"
            ]
        ),
        random_seed=42 + seed_offset,
    )


train_pairs = make_pairs("train", 0)
validation_pairs = make_pairs("validation", 1)
test_pairs = make_pairs("test", 2)

print("Writers by split:")
for split_name, split_writers in writer_split.items():
    print(f"- {split_name}: {len(split_writers)}")
print("Pairs:")
print("- train:", len(train_pairs))
print("- validation:", len(validation_pairs))
print("- test:", len(test_pairs))


Writers by split:
- train: 208
- validation: 26
- test: 26
Pairs:
- train: 16640
- validation: 2080
- test: 2080


In [ ]:
# ## Training Cell 10 — Train and persist the best Siamese checkpoint
#
# The checkpoint and history are written directly to the versioned Drive
# directory. If Colab disconnects, the best completed epoch remains available.
# The current trainer starts a new optimization run when this cell is rerun; it
# does not restore optimizer state.

# %%
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

siamese_model = SiameseResNet18(
    embedding_dimension=int(
        verification_config["embedding_dimension"]
    ),
    use_pretrained_weights=True,
)

SIAMESE_VERSIONED_CHECKPOINT = (
    DRIVE_TRAINING_ROOT / "siamese_resnet18.pt"
)
SIAMESE_HISTORY_PATH = (
    DRIVE_TRAINING_ROOT
    / "siamese_training_history.json"
)

training_history = train_siamese_model(
    model=siamese_model,
    train_pairs=train_pairs,
    validation_pairs=validation_pairs,
    verification_config=verification_config,
    checkpoint_path=SIAMESE_VERSIONED_CHECKPOINT,
    history_path=SIAMESE_HISTORY_PATH,
    device=DEVICE,
)

print("Best Siamese checkpoint:", SIAMESE_VERSIONED_CHECKPOINT)
print("History:", SIAMESE_HISTORY_PATH)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s]


Epoch 01/10 | train 0.06015 | validation 0.05206
Epoch 02/10 | train 0.03557 | validation 0.04919
Epoch 03/10 | train 0.02438 | validation 0.06070
Epoch 04/10 | train 0.01976 | validation 0.06022
Epoch 05/10 | train 0.01573 | validation 0.07232
Epoch 06/10 | train 0.01327 | validation 0.08503
Epoch 07/10 | train 0.01152 | validation 0.07011
Epoch 08/10 | train 0.01046 | validation 0.07546
Epoch 09/10 | train 0.00913 | validation 0.07539
Epoch 10/10 | train 0.00820 | validation 0.07800
Best Siamese checkpoint: /content/drive/MyDrive/Signature-Forensic-Prototype/models/training_runs/20260729T151920_289023Z/siamese_resnet18.pt
History: /content/drive/MyDrive/Signature-Forensic-Prototype/models/training_runs/20260729T151920_289023Z/siamese_training_history.json


In [ ]:
# ## Training Cell 11 — Calibrate on validation writers and test once
#
# Threshold selection uses only validation writers. Held-out test scores describe
# performance and must not be used to tune the saved threshold.

import numpy as np
from sklearn.metrics import roc_auc_score

best_siamese_model, checkpoint_metadata = (
    load_siamese_checkpoint(
        SIAMESE_VERSIONED_CHECKPOINT,
        DEVICE,
    )
)

validation_scores, validation_labels = collect_pair_scores(
    best_siamese_model,
    validation_pairs,
    verification_config,
    DEVICE,
)
verification_calibration = calibrate_similarity_thresholds(
    validation_scores,
    validation_labels,
    uncertainty_margin=float(
        verification_config["uncertainty_margin"]
    ),
)

CALIBRATION_VERSIONED_PATH = (
    DRIVE_TRAINING_ROOT
    / "verification_calibration.json"
)
save_calibration(
    verification_calibration,
    CALIBRATION_VERSIONED_PATH,
)

test_scores, test_labels = collect_pair_scores(
    best_siamese_model,
    test_pairs,
    verification_config,
    DEVICE,
)
fixed_threshold = float(
    verification_calibration["eer_threshold"]
)
test_predictions = (
    test_scores >= fixed_threshold
).astype(np.int64)
test_positive = test_labels == 1
test_negative = test_labels == 0

bhsig_test_report = {
    "evaluation_type": "held_out_writer_test",
    "threshold_source": "validation_writers",
    "pair_count": int(len(test_scores)),
    "fixed_threshold": round(fixed_threshold, 6),
    "roc_auc": round(
        float(
            roc_auc_score(
                test_labels,
                test_scores,
            )
        ),
        6,
    ),
    "accuracy_at_validation_threshold": round(
        float(np.mean(test_predictions == test_labels)),
        6,
    ),
    "false_acceptance_rate": round(
        float(
            np.mean(
                test_predictions[test_negative] == 1
            )
        ),
        6,
    ),
    "false_rejection_rate": round(
        float(
            np.mean(
                test_predictions[test_positive] == 0
            )
        ),
        6,
    ),
    "checkpoint_epoch": int(
        checkpoint_metadata["epoch"]
    ),
}

BHSIG_TEST_REPORT_PATH = (
    DRIVE_TRAINING_ROOT
    / "bhsig260_held_out_test_report.json"
)
BHSIG_TEST_REPORT_PATH.write_text(
    json.dumps(
        bhsig_test_report,
        indent=2,
    ),
    encoding="utf-8",
)

print("Calibration:")
print(json.dumps(verification_calibration, indent=2))
print("Held-out test:")
print(json.dumps(bhsig_test_report, indent=2))

Calibration:
{
  "roc_auc": 0.956391,
  "equal_error_rate": 0.110577,
  "eer_threshold": 0.887161,
  "uncertainty_margin": 0.05,
  "elevated_inconsistency_below": 0.837161,
  "provisionally_consistent_at_or_above": 0.937161,
  "positive_score_mean": 0.94849,
  "negative_score_mean": 0.621981,
  "calibration_pair_count": 2080
}
Held-out test:
{
  "evaluation_type": "held_out_writer_test",
  "threshold_source": "validation_writers",
  "pair_count": 2080,
  "fixed_threshold": 0.887161,
  "roc_auc": 0.965143,
  "accuracy_at_validation_threshold": 0.902885,
  "false_acceptance_rate": 0.098077,
  "false_rejection_rate": 0.096154,
  "checkpoint_epoch": 2
}


In [ ]:
# ## Training Cell 12 — Optionally promote the reviewed Siamese artifacts
#
# Promote checkpoint and calibration together. Mixing a checkpoint with
# calibration from another training run produces invalid interpretation.

# PROMOTE_SIAMESE = True to save the checkpoint. To skip keep as False.
PROMOTE_SIAMESE = False

CANONICAL_SIAMESE_CHECKPOINT = (
    PROJECT_DIRECTORY
    / verification_config["model_path"]
)
CANONICAL_CALIBRATION_PATH = (
    PROJECT_DIRECTORY
    / verification_config["calibration_path"]
)

if PROMOTE_SIAMESE:
    CANONICAL_SIAMESE_CHECKPOINT.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    for canonical, versioned, backup_name in (
        (
            CANONICAL_SIAMESE_CHECKPOINT,
            SIAMESE_VERSIONED_CHECKPOINT,
            "previous_siamese_resnet18.pt",
        ),
        (
            CANONICAL_CALIBRATION_PATH,
            CALIBRATION_VERSIONED_PATH,
            "previous_verification_calibration.json",
        ),
    ):
        if canonical.is_file():
            backup = (
                DRIVE_TRAINING_ROOT / backup_name
            )
            shutil.copy2(canonical, backup)
            print("Backed up:", backup)
        shutil.copy2(versioned, canonical)
        print("Promoted:", canonical)

    # These files are not required for inference, but keeping them beside the
    # promoted checkpoint preserves how it was trained and evaluated.
    supporting_promotions = {
        PROJECT_DIRECTORY
        / verification_config["writer_split_path"]: WRITER_SPLIT_PATH,
        PROJECT_DIRECTORY
        / verification_config["training_history_path"]: SIAMESE_HISTORY_PATH,
        PROJECT_DIRECTORY
        / "models/bhsig260_held_out_test_report.json": BHSIG_TEST_REPORT_PATH,
    }
    for canonical, versioned in supporting_promotions.items():
        canonical.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(versioned, canonical)
        print("Promoted supporting artifact:", canonical)
else:
    print(
        "Siamese artifacts were not promoted. Review:",
        DRIVE_TRAINING_ROOT,
    )

Promoted: /content/drive/MyDrive/Signature-Forensic-Prototype/models/siamese_resnet18.pt
Promoted: /content/drive/MyDrive/Signature-Forensic-Prototype/models/verification_calibration.json
Promoted supporting artifact: /content/drive/MyDrive/Signature-Forensic-Prototype/models/writer_split.json
Promoted supporting artifact: /content/drive/MyDrive/Signature-Forensic-Prototype/models/siamese_training_history.json
Promoted supporting artifact: /content/drive/MyDrive/Signature-Forensic-Prototype/models/bhsig260_held_out_test_report.json


In [ ]:
# # Part C — Optional untouched CEDAR validation
#
# Run this part after training and calibration. CEDAR must not influence model
# weights, BHSig pair construction, or threshold selection.
#
# ## Training Cell 13 — Stage CEDAR, evaluate, and optionally promote report
# Run with RUN_CEDAR_EXTERNAL_VALIDATION = True and PROMOTE_CEDAR_REPORT = False if you just want to validate but dont want to save and use the result later
# If you want to use then run with both Flag = True
RUN_CEDAR_EXTERNAL_VALIDATION = True
PROMOTE_CEDAR_REPORT = True

if RUN_CEDAR_EXTERNAL_VALIDATION:

    # Cell 8 extracted the complete BHSig260.zip archive here.
    # Since that archive also contains CEDAR, reuse the existing extraction.
    CEDAR_EXTRACT_ROOT = BHSIG_EXTRACT_ROOT

    if not CEDAR_EXTRACT_ROOT.is_dir():
        raise FileNotFoundError(
            "The extracted dataset directory is unavailable. "
            "Run Training Cell 8 first in this Colab runtime."
        )

    # Search recursively, so CEDAR may be nested at any depth.
    cedar_writers = discover_cedar(CEDAR_EXTRACT_ROOT)

    print(
        "CEDAR loaded from the existing BHSig260 extraction:",
        CEDAR_EXTRACT_ROOT,
    )
    print("CEDAR writers discovered:", len(cedar_writers))

    cedar_external_report = evaluate_cedar_external(
        model=best_siamese_model,
        writers=cedar_writers,
        verification_config=verification_config,
        calibration=verification_calibration,
        device=DEVICE,
        pairs_per_class_per_writer=120,
        random_seed=42,
    )

    CEDAR_VERSIONED_REPORT_PATH = (
        DRIVE_TRAINING_ROOT
        / "cedar_external_test_report.json"
    )

    CEDAR_VERSIONED_REPORT_PATH.write_text(
        json.dumps(
            cedar_external_report,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("Untouched CEDAR report:")
    print(
        json.dumps(
            cedar_external_report,
            indent=2,
        )
    )

    if PROMOTE_CEDAR_REPORT:
        canonical_cedar_report = (
            PROJECT_DIRECTORY
            / "models/cedar_external_test_report.json"
        )

        if canonical_cedar_report.is_file():
            backup = (
                DRIVE_TRAINING_ROOT
                / "previous_cedar_external_test_report.json"
            )
            shutil.copy2(
                canonical_cedar_report,
                backup,
            )
            print("Previous CEDAR report backed up:", backup)

        shutil.copy2(
            CEDAR_VERSIONED_REPORT_PATH,
            canonical_cedar_report,
        )
        print(
            "CEDAR report promoted:",
            canonical_cedar_report,
        )
    else:
        print(
            "CEDAR report was not promoted. Review:",
            CEDAR_VERSIONED_REPORT_PATH,
        )
else:
    print("CEDAR external validation skipped.")

CEDAR loaded from the existing BHSig260 extraction: /content/signature_model_training/20260729T151920_289023Z/datasets/bhsig260
CEDAR writers discovered: 55
Untouched CEDAR report:
{
  "evaluation_type": "untouched_external_test",
  "training_dataset": "BHSig260",
  "external_test_dataset": "CEDAR",
  "pair_count": 13200,
  "positive_pair_count": 6600,
  "negative_pair_count": 6600,
  "bhsig_fixed_threshold": 0.887161,
  "roc_auc": 0.782203,
  "accuracy_at_bhsig_threshold": 0.622273,
  "false_acceptance_rate_at_bhsig_threshold": 0.698182,
  "false_rejection_rate_at_bhsig_threshold": 0.057273,
  "genuine_score_mean": 0.963321,
  "forgery_score_mean": 0.901509,
  "cedar_descriptive_eer": 0.291212,
  "cedar_descriptive_eer_threshold": 0.961084,
  "threshold_note": "Operational metrics use the untouched BHSig260 validation threshold. The CEDAR EER threshold is descriptive only.",
  "dataset_summary": {
    "writers": 55,
    "genuine_images": 1320,
    "forged_images": 1320
  }
}
Previous 

In [ ]:
# ## Training Cell 14 — Save hashes and the training manifest
#
# Hashes link promoted artifacts to this exact run. They do not assess model
# quality; they detect accidental file replacement.

# %%
import hashlib


def sha256_file(path):
    """Return a reproducible SHA-256 digest for one persistent artifact."""

    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


artifact_paths = {
    "yolo_checkpoint": YOLO_VERSIONED_CHECKPOINT,
    "yolo_validation": YOLO_VALIDATION_REPORT_PATH,
    "siamese_checkpoint": SIAMESE_VERSIONED_CHECKPOINT,
    "siamese_history": SIAMESE_HISTORY_PATH,
    "writer_split": WRITER_SPLIT_PATH,
    "verification_calibration": CALIBRATION_VERSIONED_PATH,
    "bhsig_test_report": BHSIG_TEST_REPORT_PATH,
}
if (
    RUN_CEDAR_EXTERNAL_VALIDATION
    and "CEDAR_VERSIONED_REPORT_PATH" in globals()
):
    artifact_paths["cedar_external_report"] = (
        CEDAR_VERSIONED_REPORT_PATH
    )

training_manifest = {
    "training_id": TRAINING_ID,
    "device": torch.cuda.get_device_name(0),
    "datasets": {
        "yolo_archive": str(YOLO_DATASET_ZIP),
        "bhsig260_archive": str(BHSIG260_ZIP),
        "cedar_archive": (
            str(CEDAR_ZIP)
            if RUN_CEDAR_EXTERNAL_VALIDATION
            else None
        ),
    },
    "artifacts": {
        name: {
            "path": str(path),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }
        for name, path in artifact_paths.items()
    },
    "promotion": {
        "yolo": PROMOTE_YOLO,
        "siamese_and_calibration": PROMOTE_SIAMESE,
        "cedar_report": (
            PROMOTE_CEDAR_REPORT
            if RUN_CEDAR_EXTERNAL_VALIDATION
            else False
        ),
    },
    "interpretation": (
        "Metrics and thresholds are prototype model-development evidence, "
        "not authenticity or forgery determinations."
    ),
}

TRAINING_MANIFEST_PATH = (
    DRIVE_TRAINING_ROOT
    / "training_manifest.json"
)
TRAINING_MANIFEST_PATH.write_text(
    json.dumps(
        training_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("MODEL TRAINING WORKFLOW COMPLETED")
print("=================================")
print("Versioned artifacts:", DRIVE_TRAINING_ROOT)
print("Manifest:", TRAINING_MANIFEST_PATH)
print(
    "\nCanonical checkpoints change only when the corresponding "
    "PROMOTE_* flag is True."
)

MODEL TRAINING WORKFLOW COMPLETED
Versioned artifacts: /content/drive/MyDrive/Signature-Forensic-Prototype/models/training_runs/20260729T151920_289023Z
Manifest: /content/drive/MyDrive/Signature-Forensic-Prototype/models/training_runs/20260729T151920_289023Z/training_manifest.json

Canonical checkpoints change only when the corresponding PROMOTE_* flag is True.
